# DeepSets vs GraphSAGE Comparison (Extrapolation)

## Objective
Compare the extrapolation performance of DeepSets and GraphSAGE models on surface code quantum error correction. The models are trained on mixed distances (d=3, 5, 7) and tested on d=3, 5, 7, 9, 11, 13 to evaluate their ability to generalize to larger code distances.

## Models Compared
1. **DeepSets**: Permutation-invariant architecture using coordinate inputs (x, y, t).
2. **GraphSAGE**: Graph Neural Network with weighted aggregation (GNN).
3. **SimpleNN**: Baseline feedforward neural network (specialist per distance).
4. **MWPM**: Minimum Weight Perfect Matching (algorithmic baseline).

## Metrics
- **Accuracy**: Prediction accuracy vs Code Distance.
- **Logical Error Rate (LER)**: $P_{logical}$ vs Code Distance.
- **Inference Speed**: Throughput and Latency scaling.
- **Parameter Efficiency**: Number of parameters and model size.

## 1. Imports and Setup

In [ ]:
import sys
import json
import time
import gc
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from scipy import stats
from scipy.stats import wilcoxon
from torch_geometric.loader import DataLoader
import stim

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set paths
BASE_PATH = Path('../..').resolve()  # code/results/deepsets_vs_gnn_comparison -> code/
if str(BASE_PATH) not in sys.path:
    sys.path.insert(0, str(BASE_PATH))

print(f"BASE_PATH: {BASE_PATH}")

# Import custom models
from benchmark_models import DeepSets, SimpleNN, SurfaceCodeSampler
from models import GraphSAGE, SparseGraph

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Configuration

In [ ]:
# Evaluation config
DISTANCES = [3, 5, 7, 9, 11, 13]  # Extrapolation range
TEST_SAMPLES = 10000              # Samples per distance for accuracy
BATCH_SIZE = 512                  # Inference batch size
Experiment_Name = "equal_333333"  # Model trained on balanced d=3,5,7 data

# Directories
RESULTS_DIR = Path("results")
PLOTS_DIR = Path("plots")
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

print(f"Experiment: {Experiment_Name}")
print(f"Distances to evaluate: {DISTANCES}")

## 3. Model Loading Functions

In [ ]:
def load_deepsets_model(experiment_name, base_path):
    """Load DeepSets extrapolation model."""
    model_path = base_path / "deepsets" / "extrapolation" / "models" / "revised_training" / f"{experiment_name}.pt"
    if not model_path.exists():
        raise FileNotFoundError(f"DeepSets model not found at {model_path}")
    
    # Initialize dummy model to load weights
    model = DeepSets(base_path=base_path, device=device)
    model.load(str(model_path))
    model.model.eval()
    return model

def load_graphsage_model(experiment_name, base_path):
    """Load GraphSAGE extrapolation model."""
    model_path = base_path / "gSAGE" / "extrapolation" / "models" / "revised_training" / f"{experiment_name}.pt"
    if not model_path.exists():
        raise FileNotFoundError(f"GraphSAGE model not found at {model_path}")
    
    # Initialize dummy model to load weights
    model = GraphSAGE(base_path=base_path, device=device)
    model.load(str(model_path))
    model.model.eval()
    return model

def load_simplenn_model(distance, base_path):
    """Load SimpleNN specialist model for a specific distance."""
    model_path = base_path / "nn" / "training" / "models" / "revised_training" / f"d{distance}_specialist.pt"
    if not model_path.exists():
        print(f"Warning: SimpleNN model for d={distance} not found.")
        return None
    
    # Note: SimpleNN init usually requires num_detectors, but .load() might handle it or we guess
    # For simplicity, we assume the class handles loading if we just instantiate
    # But benchmark_models.py SimpleNN might need args. Let's try to load checkpoint to peek args.
    checkpoint = torch.load(model_path, map_location='cpu')
    num_detectors = checkpoint.get('num_detectors', 0)
    if num_detectors == 0:
         # Fallback calculation
         circuit = stim.Circuit.generated("surface_code:rotated_memory_z", rounds=distance, distance=distance, after_clifford_depolarization=0.001)
         num_detectors = circuit.num_detectors

    model = SimpleNN(nickname=f"d{distance}", in_channels=num_detectors, base_path=base_path, device=device)
    try:
        model.load(str(model_path))
    except: 
        model.model.load_state_dict(checkpoint['state_dict'], strict=False)
    
    model.model.eval()
    return model

def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

## 4. Benchmarking Logic

In [ ]:
def benchmark_models(distances, experiment_name):
    results = {
        'distance': [],
        'model': [],
        'accuracy': [],
        'ler': [],
        'inference_time_ms': [],
        'params': []
    }
    
    # Load extrapolation models (one instance for all distances)
    print("Loading DeepSets Model...")
    deepsets = load_deepsets_model(experiment_name, BASE_PATH)
    ds_params = count_params(deepsets.model)
    
    print("Loading GraphSAGE Model...")
    gsage = load_graphsage_model(experiment_name, BASE_PATH)
    gs_params = count_params(gsage.model)
    
    sampler = SurfaceCodeSampler(device=device)
    
    for d in tqdm(distances, desc="Evaluating Distances"):
        # Generate Data
        detections, labels = sampler.sample(d=d, num_samples=TEST_SAMPLES, p_values=[0.01], return_p_labels=False)
        # Store numpy version for accuracy calculation, keep original tensor for PyG conversion
        labels_np = labels.cpu().numpy()
        
        # --- DeepSets ---
        start_time = time.time()
        ds_probs = deepsets.predict(detections, d).cpu().numpy()
        ds_time = (time.time() - start_time) / TEST_SAMPLES * 1000  # ms per sample
        ds_preds = (ds_probs > 0.5).astype(int)
        ds_acc = np.mean(ds_preds.flatten() == labels_np)
        ds_ler = 1 - ds_acc
        
        results['distance'].append(d)
        results['model'].append('DeepSets')
        results['accuracy'].append(ds_acc)
        results['ler'].append(ds_ler)
        results['inference_time_ms'].append(ds_time)
        results['params'].append(ds_params)
        
        # --- GraphSAGE ---
        # Create PyG graphs
        # Note: We iterate over tensor inputs so that GraphSAGE conversion works correctly
        graph_builder = SparseGraph(device=device)
        graphs = [graph_builder.to_pyg(det, lbl) for det, lbl in zip(detections, labels)]
        loader = DataLoader(graphs, batch_size=BATCH_SIZE, shuffle=False)
        
        start_time = time.time()
        gs_preds_list = []
        gsage.model.eval()
        with torch.no_grad():
            for batch in loader:
                batch = batch.to(device)
                out = gsage.model(batch)
                gs_preds_list.extend(out.cpu().numpy().flatten())
        gs_time = (time.time() - start_time) / TEST_SAMPLES * 1000
        gs_probs = np.array(gs_preds_list)
        gs_binary = (gs_probs > 0.5).astype(int)
        gs_acc = np.mean(gs_binary == labels_np)
        gs_ler = 1 - gs_acc
        
        results['distance'].append(d)
        results['model'].append('GraphSAGE')
        results['accuracy'].append(gs_acc)
        results['ler'].append(gs_ler)
        results['inference_time_ms'].append(gs_time)
        results['params'].append(gs_params)
        
        # --- SimpleNN (Baseline) ---
        simplenn = load_simplenn_model(d, BASE_PATH)
        if simplenn:
            s_params = count_params(simplenn.model)
            start_time = time.time()
            sn_probs = simplenn.predict(detections).cpu().numpy()
            sn_time = (time.time() - start_time) / TEST_SAMPLES * 1000
            sn_preds = (sn_probs > 0.5).astype(int)
            sn_acc = np.mean(sn_preds.flatten() == labels_np)
            sn_ler = 1 - sn_acc
            
            results['distance'].append(d)
            results['model'].append('SimpleNN')
            results['accuracy'].append(sn_acc)
            results['ler'].append(sn_ler)
            results['inference_time_ms'].append(sn_time)
            results['params'].append(s_params)
            
    return pd.DataFrame(results)

## 5. Run Comparison

In [ ]:
print("Starting benchmarking...")
df_results = benchmark_models(DISTANCES, Experiment_Name)

# Save Results
df_results.to_csv(RESULTS_DIR / "comparison_results.csv", index=False)
display(df_results.groupby('model').mean(numeric_only=True))

## 6. Visualization

In [ ]:
# Plot Accuracy
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_results, x='distance', y='accuracy', hue='model', style='model', markers=True, markersize=8)
plt.title(f'Accuracy vs Code Distance ({Experiment_Name})')
plt.ylabel('Accuracy')
plt.xlabel('Code Distance (d)')
plt.xticks(DISTANCES)
plt.savefig(PLOTS_DIR / 'accuracy_vs_distance.png', dpi=300)
plt.show()

# Plot Logical Error Rate (Log Scale)
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_results, x='distance', y='ler', hue='model', style='model', markers=True, markersize=8)
plt.yscale('log')
plt.title(f'Logical Error Rate vs Distance ({Experiment_Name})')
plt.ylabel('Logical Error Rate (LER)')
plt.xlabel('Code Distance (d)')
plt.xticks(DISTANCES)
plt.savefig(PLOTS_DIR / 'ler_vs_distance.png', dpi=300)
plt.show()

# Plot Inference Time
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_results, x='distance', y='inference_time_ms', hue='model', style='model', markers=True, markersize=8)
plt.title('Inference Speed Scaling')
plt.ylabel('Time per Sample (ms)')
plt.xlabel('Code Distance (d)')
plt.xticks(DISTANCES)
plt.savefig(PLOTS_DIR / 'inference_time.png', dpi=300)
plt.show()